In [ ]:
%pip install torch matplotlib scikit-learn shap vitaldb


In [ ]:
from __future__ import annotations
import gc, logging, math, os, pickle, random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import vitaldb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

torch.manual_seed(42); np.random.seed(42); random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
CONFIG: Dict[str, Any] = {
    'model_id': 5, 'model_name': 'PatientContextMLP',
    'output_file': 'model_5.py', 'log_file': 'model_5.log',
    'checkpoint_path': 'model_5_best.pt',
    'shap_output': 'shap_model_5.png', 'pred_output': 'prediction_model_5.png',
    'caseid_min': 1, 'caseid_max': 6388,
    'clinical_params': [
        'caseid', 'age', 'sex', 'bmi', 'asa', 'emop',
        'ane_type', 'optype', 'department', 'position',
        'preop_htn', 'preop_dm', 'anestart', 'aneend', 'opstart', 'opend',
    ],
    'lab_params': ['alb', 'hb', 'gluc'],
    'continuous_features': ['age', 'bmi', 'preop_alb', 'preop_hb', 'preop_gluc'],
    'ordinal_features': ['asa'],
    'binary_features': ['sex', 'emop', 'preop_htn', 'preop_dm'],
    'onehot_features': ['ane_type', 'optype', 'department', 'position'],
    'et_min_sec': 0.0, 'et_max_sec': 3600.0,
    'embedding_dim': 128, 'hidden_dims': [64, 128, 64], 'dropout': 0.3,
    'batch_size': 256, 'learning_rate': 1e-3, 'weight_decay': 1e-4,
    'max_epochs': 100, 'early_stopping_patience': 10, 'grad_clip_norm': 1.0,
    'val_frac': 0.10, 'test_frac': 0.10, 'random_seed': 42,
    'lab_missing_strategy': 'median', 'cat_missing_strategy': 'mode',
    'min_case_duration_sec': 1800,
}


In [ ]:
def setup_logging(log_file: str) -> logging.Logger:
    logger = logging.getLogger('model_5')
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(name)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    fh = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    fh.setLevel(logging.DEBUG); fh.setFormatter(fmt); logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO); ch.setFormatter(fmt); logger.addHandler(ch)
    return logger

LOGGER = setup_logging(CONFIG['log_file'])


In [ ]:
def load_clinical_metadata(config):
    all_caseids = list(range(config['caseid_min'], config['caseid_max'] + 1))
    LOGGER.info('Loading clinical metadata for %d cases ...', len(all_caseids))
    try:
        df = vitaldb.load_clinical_data(caseids=all_caseids, params=config['clinical_params'])
    except Exception as exc:
        LOGGER.error('load_clinical_data() failed: %s', exc); raise
    LOGGER.info('Clinical metadata loaded: %d rows, %d cols', len(df), len(df.columns))
    return df

def load_lab_data_wide(config):
    all_caseids = list(range(config['caseid_min'], config['caseid_max'] + 1))
    LOGGER.info('Loading lab data for %d cases ...', len(all_caseids))
    try:
        df_long = vitaldb.load_lab_data(caseids=all_caseids, params=config['lab_params'])
    except Exception as exc:
        LOGGER.error('load_lab_data() failed: %s', exc); raise
    if df_long is None or len(df_long) == 0:
        LOGGER.warning('load_lab_data() returned empty DataFrame')
        return pd.DataFrame({'caseid': all_caseids})
    df_long = df_long.dropna(subset=['result'])
    df_long['result'] = pd.to_numeric(df_long['result'], errors='coerce')
    df_preop = df_long.sort_values('dt').groupby(['caseid', 'name'], as_index=False).first()
    df_wide = df_preop.pivot_table(index='caseid', columns='name', values='result', aggfunc='first').reset_index()
    df_wide.columns = ['caseid' if c == 'caseid' else f'preop_{c}' for c in df_wide.columns]
    LOGGER.info('Lab data wide shape: %s', df_wide.shape)
    return df_wide


In [ ]:
_PHYS_BOUNDS = {
    'age': (0.0, 120.0), 'bmi': (10.0, 80.0),
    'preop_alb': (1.0, 6.0), 'preop_hb': (3.0, 22.0), 'preop_gluc': (30.0, 600.0),
}

def remove_outliers(df):
    df = df.copy()
    for col, (lo, hi) in _PHYS_BOUNDS.items():
        if col not in df.columns: continue
        n_before = df[col].notna().sum()
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
        n_removed = n_before - df[col].notna().sum()
        if n_removed: LOGGER.debug('Outlier removal: %s — set %d values to NaN', col, n_removed)
    return df

def build_et_labels(df, config):
    df = df.copy()
    df['emergence_sec'] = pd.to_numeric(df['aneend'], errors='coerce') - pd.to_numeric(df['opend'], errors='coerce')
    valid_mask = df['emergence_sec'].notna() & df['emergence_sec'].between(config['et_min_sec'], config['et_max_sec'], inclusive='neither')
    df.loc[~valid_mask, 'emergence_sec'] = np.nan
    df['log_emergence_sec'] = np.log(df['emergence_sec'])
    LOGGER.info('ET labels: %d / %d cases valid in (%.0f, %.0f) s', valid_mask.sum(), len(df), config['et_min_sec'], config['et_max_sec'])
    return df

def apply_eligibility_filter(df, config):
    n_before = len(df)
    df = df.dropna(subset=['caseid']).copy()
    df['caseid'] = df['caseid'].astype(int)
    duration = pd.to_numeric(df['aneend'], errors='coerce') - pd.to_numeric(df['anestart'], errors='coerce')
    df = df[duration >= config['min_case_duration_sec']].copy()
    LOGGER.info('Eligibility filter: %d → %d cases (removed %d)', n_before, len(df), n_before - len(df))
    return df

def split_cases(all_caseids, val_frac=0.10, test_frac=0.10, seed=42):
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    train, val = train_test_split(train_val, test_size=val_frac / (1.0 - test_frac), random_state=seed)
    LOGGER.info('Split: train=%d, val=%d, test=%d', len(train), len(val), len(test))
    return train, val, test


In [ ]:
def build_feature_matrix(df, config, fit_mode, artifacts=None):
    df = df.copy()
    if artifacts is None: artifacts = {}

    for col in config['continuous_features']:
        if col not in df.columns: df[col] = np.nan
        if fit_mode: artifacts[f'median_{col}'] = float(df[col].median()) if df[col].notna().any() else 0.0
        df[col] = df[col].fillna(artifacts[f'median_{col}'])

    for col in config['ordinal_features'] + config['binary_features']:
        if col not in df.columns: df[col] = np.nan
        if fit_mode: artifacts[f'mode_{col}'] = float(df[col].mode().iloc[0]) if df[col].notna().any() else (2.0 if col in config['ordinal_features'] else 0.0)
        df[col] = df[col].fillna(artifacts[f'mode_{col}']).astype(float)

    onehot_frames = []
    for col in config['onehot_features']:
        if col not in df.columns: df[col] = 'unknown'
        df[col] = df[col].fillna('unknown').astype(str)
        if fit_mode: artifacts[f'onehot_cats_{col}'] = sorted(df[col].unique().tolist())
        cats = artifacts[f'onehot_cats_{col}']
        dummies = pd.get_dummies(df[col], prefix=col)
        for ec in [f'{col}_{c}' for c in cats]:
            if ec not in dummies.columns: dummies[ec] = 0
        onehot_frames.append(dummies[[f'{col}_{c}' for c in cats]].reset_index(drop=True))

    scalar_cols = config['continuous_features'] + config['ordinal_features'] + config['binary_features']
    df_combined = pd.concat([df[scalar_cols].reset_index(drop=True).astype(float)] + onehot_frames, axis=1)
    feature_names = list(df_combined.columns)
    X = df_combined.values.astype(np.float32)

    cont_indices = [feature_names.index(c) for c in config['continuous_features']]
    if fit_mode:
        scaler = StandardScaler()
        X[:, cont_indices] = scaler.fit_transform(X[:, cont_indices])
        artifacts['scaler'] = scaler
        artifacts['feature_names'] = feature_names
        LOGGER.info('Fitted scaler on %d training samples; %d features total', len(X), len(feature_names))
    else:
        X[:, cont_indices] = artifacts['scaler'].transform(X[:, cont_indices])

    nan_count = np.isnan(X).sum()
    if nan_count: LOGGER.warning('Replacing %d residual NaN values with 0', nan_count)
    X = np.nan_to_num(X, nan=0.0)
    return X, feature_names, artifacts


In [ ]:
class Model5Dataset(Dataset):
    def __init__(self, X, caseids, et_labels):
        assert len(X) == len(caseids) == len(et_labels)
        self.X = torch.from_numpy(X.astype(np.float32))
        self.caseids = caseids
        self.et_labels = torch.from_numpy(et_labels.astype(np.float32))

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return {
            'x': self.X[idx],
            'label_ioh_t0':   torch.tensor(-1.0, dtype=torch.float32),
            'label_ioh_t30':  torch.tensor(-1.0, dtype=torch.float32),
            'label_ioh_t60':  torch.tensor(-1.0, dtype=torch.float32),
            'label_ioh_t180': torch.tensor(-1.0, dtype=torch.float32),
            'label_ioh_t300': torch.tensor(-1.0, dtype=torch.float32),
            'label_nh':       torch.tensor(-1,   dtype=torch.long),
            'label_et':       self.et_labels[idx],
            'caseid':         self.caseids[idx],
            'window_start':   -1,
            'sqi_flag':       torch.ones(1, dtype=torch.float32),
            'modality_present': torch.tensor(True, dtype=torch.bool),
        }


In [ ]:
def build_dataloaders(train_ds, val_ds, test_ds, config):
    kw = dict(batch_size=config['batch_size'], pin_memory=True)
    train_loader = DataLoader(train_ds, shuffle=True,  num_workers=4, drop_last=False, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, num_workers=2, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, num_workers=2, **kw)
    LOGGER.info('DataLoaders: train=%d batches, val=%d batches, test=%d batches',
                len(train_loader), len(val_loader), len(test_loader))
    return train_loader, val_loader, test_loader


In [ ]:
class MLPBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True), nn.Dropout(p=dropout),
        )
    def forward(self, x): return self.block(x)


class PatientContextMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, embedding_dim=128, dropout=0.3):
        super().__init__()
        assert len(hidden_dims) == 3
        h1, h2, h3 = hidden_dims
        self.layer1 = MLPBlock(input_dim, h1, dropout)
        self.layer2 = MLPBlock(h1, h2, dropout)
        self.layer3 = MLPBlock(h2, h3, dropout)
        self.output_proj = nn.Linear(h3, embedding_dim)
        self.et_regression_head = nn.Linear(embedding_dim, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.layer3(self.layer2(self.layer1(x)))
        embedding = self.output_proj(h)
        return embedding, self.et_regression_head(embedding)


In [ ]:
def et_lognormal_loss(pred_log, target_log):
    pred_log = pred_log.squeeze(-1)
    valid_mask = ~torch.isnan(target_log)
    if not valid_mask.any():
        return torch.tensor(0.0, device=pred_log.device, requires_grad=True)
    return nn.functional.mse_loss(pred_log[valid_mask], target_log[valid_mask])


def compute_metrics(preds_log, targets_log):
    valid = ~np.isnan(targets_log)
    if not valid.any():
        return {k: np.nan for k in ['mae_sec', 'rmse_sec', 'mdape_pct', 'mse_log']}
    p, t = preds_log[valid], targets_log[valid]
    p_sec, t_sec = np.exp(p), np.exp(t)
    return {
        'mse_log':   float(np.mean((p - t) ** 2)),
        'mae_sec':   float(np.mean(np.abs(p_sec - t_sec))),
        'rmse_sec':  float(np.sqrt(np.mean((p_sec - t_sec) ** 2))),
        'mdape_pct': float(np.median(np.abs((p_sec - t_sec) / (t_sec + 1e-6)) * 100)),
    }


In [ ]:
def train_one_epoch(model, loader, optimizer, amp_scaler, device, config):
    model.train()
    total_loss, n = 0.0, 0
    for batch in tqdm(loader, desc='  Train', leave=False):
        x = batch['x'].to(device, non_blocking=True)
        et_target = batch['label_et'].to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            _, et_pred = model(x)
            loss = et_lognormal_loss(et_pred, et_target)
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
        amp_scaler.step(optimizer); amp_scaler.update()
        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1)


def validate(model, loader, device, config):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  Val  ', leave=False):
            x = batch['x'].to(device, non_blocking=True)
            et_target = batch['label_et'].to(device, non_blocking=True)
            with autocast(device_type='cuda'):
                _, et_pred = model(x)
                loss = et_lognormal_loss(et_pred, et_target)
            total_loss += loss.item(); n += 1
            all_preds.append(et_pred.squeeze(-1).cpu().numpy())
            all_targets.append(et_target.cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_targets))
    return total_loss / max(n, 1), metrics


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, checkpoint_path='best.pt'):
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss = math.inf
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss; self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            LOGGER.debug('EarlyStopping: new best %.5f → saved', val_loss)
        else:
            self.counter += 1
            LOGGER.debug('EarlyStopping: no improvement (%d/%d)', self.counter, self.patience)
            if self.counter >= self.patience:
                self.should_stop = True
                LOGGER.info('EarlyStopping triggered after %d epochs', self.patience)


def train_model(model, train_loader, val_loader, device, config):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['max_epochs'], eta_min=config['learning_rate'] * 0.01)
    amp_scaler = GradScaler(device_type='cuda')
    es = EarlyStopping(patience=config['early_stopping_patience'], checkpoint_path=config['checkpoint_path'])
    LOGGER.info('Starting training — max_epochs=%d, patience=%d', config['max_epochs'], config['early_stopping_patience'])
    for epoch in range(1, config['max_epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, amp_scaler, device, config)
        val_loss, val_metrics = validate(model, val_loader, device, config)
        scheduler.step()
        LOGGER.info('Epoch %03d | train=%.5f | val=%.5f | mae=%.1fs | rmse=%.1fs | mdape=%.1f%%',
                    epoch, train_loss, val_loss,
                    val_metrics.get('mae_sec', float('nan')),
                    val_metrics.get('rmse_sec', float('nan')),
                    val_metrics.get('mdape_pct', float('nan')))
        es.step(val_loss, model)
        if es.should_stop:
            LOGGER.info('Early stopping at epoch %d', epoch); break
    LOGGER.info('Loading best checkpoint from %s', config['checkpoint_path'])
    model.load_state_dict(torch.load(config['checkpoint_path'], map_location=device, weights_only=True))
    return model


In [ ]:
def evaluate_test(model, test_loader, device, config):
    LOGGER.info('─── Test Evaluation ───────────────────────────────────────')
    test_loss, metrics = validate(model, test_loader, device, config)
    LOGGER.info('Test loss (log-normal MSE): %.5f', test_loss)
    for k, v in metrics.items(): LOGGER.info('  %s: %.4f', k, v)
    print('\n════ Test Set Metrics (Model 5 — Patient Context MLP) ════')
    print(f"  Log-space MSE : {test_loss:.5f}")
    print(f"  MSE (log)     : {metrics.get('mse_log', float('nan')):.5f}")
    print(f"  MAE (s)       : {metrics.get('mae_sec', float('nan')):.2f}")
    print(f"  RMSE (s)      : {metrics.get('rmse_sec', float('nan')):.2f}")
    print(f"  MdAPE         : {metrics.get('mdape_pct', float('nan')):.2f} %")
    print('═══════════════════════════════════════════════════════════\n')


def explain_with_shap(model, test_loader, feature_names, device, config):
    LOGGER.info('Computing SHAP values for Model 5 ...')
    model.eval()
    X_all = torch.cat([b['x'] for b in test_loader], dim=0).to(device)
    background = X_all[:min(100, len(X_all))]

    class _ETWrapper(nn.Module):
        def __init__(self, base): super().__init__(); self.base = base
        def forward(self, x): _, et = self.base(x); return et

    try:
        explainer = shap.DeepExplainer(_ETWrapper(model), background)
        n_explain = min(200, len(X_all))
        sv = explainer.shap_values(X_all[:n_explain])
        sv = np.array(sv[0] if isinstance(sv, list) else sv).squeeze()
        plt.figure(figsize=(12, 8))
        shap.summary_plot(sv, X_all[:n_explain].cpu().numpy(), feature_names=feature_names, show=False, max_display=20)
        plt.title('SHAP Feature Importance — Model 5 (Emergence Timing)')
        plt.tight_layout()
        plt.savefig(config['shap_output'], dpi=150, bbox_inches='tight'); plt.close()
        LOGGER.info('SHAP plot saved to %s', config['shap_output'])
    except Exception as exc:
        LOGGER.warning('SHAP computation failed: %s — skipping', exc)


def plot_predictions(model, test_loader, device, config):
    LOGGER.info('Generating prediction plots for Model 5 ...')
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            x = batch['x'].to(device, non_blocking=True)
            with autocast(device_type='cuda'):
                _, et_pred = model(x)
            all_preds.append(et_pred.squeeze(-1).cpu().numpy())
            all_targets.append(batch['label_et'].cpu().numpy())
    preds_log = np.concatenate(all_preds)
    targets_log = np.concatenate(all_targets)
    valid = ~np.isnan(targets_log)
    p, t = np.exp(preds_log[valid]), np.exp(targets_log[valid])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Model 5 — Patient Context MLP\nEmergence Timing Predictions (Test Set)', fontsize=13)
    lim = max(t.max(), p.max()) * 1.05
    axes[0].scatter(t, p, alpha=0.3, s=10, color='steelblue', label='Predictions')
    axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
    axes[0].set(xlabel='Actual (s)', ylabel='Predicted (s)', title='Predicted vs Actual', xlim=(0,lim), ylim=(0,lim))
    axes[0].legend(fontsize=8)
    axes[1].hist(p - t, bins=50, color='coral', edgecolor='black', lw=0.5)
    axes[1].axvline(0, color='black', lw=1.5, ls='--')
    axes[1].set(xlabel='Residual (predicted − actual, s)', ylabel='Count', title='Residual Distribution')
    plt.tight_layout()
    plt.savefig(config['pred_output'], dpi=150, bbox_inches='tight'); plt.close()
    LOGGER.info('Prediction plot saved to %s', config['pred_output'])


In [ ]:
def main():
    LOGGER.info('══════════════════════════════════════════════════════════')
    LOGGER.info('Model 5 — Patient Context MLP — pipeline start')
    LOGGER.info('══════════════════════════════════════════════════════════')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    LOGGER.info('Device: %s', device)

    # 1. Load & merge data
    df = pd.merge(load_clinical_metadata(CONFIG), load_lab_data_wide(CONFIG), on='caseid', how='left')
    LOGGER.info('Merged DataFrame: %d rows', len(df))

    # 2-3. Filter, clean, label
    df = build_et_labels(remove_outliers(apply_eligibility_filter(df, CONFIG)), CONFIG)

    # 4. Split
    train_ids, val_ids, test_ids = split_cases(df['caseid'].tolist(), CONFIG['val_frac'], CONFIG['test_frac'], CONFIG['random_seed'])
    df_train = df[df['caseid'].isin(train_ids)].reset_index(drop=True)
    df_val   = df[df['caseid'].isin(val_ids)].reset_index(drop=True)
    df_test  = df[df['caseid'].isin(test_ids)].reset_index(drop=True)
    LOGGER.info('Data sizes — train: %d, val: %d, test: %d', len(df_train), len(df_val), len(df_test))

    # 5. Feature matrices
    X_train, feature_names, artifacts = build_feature_matrix(df_train, CONFIG, fit_mode=True)
    X_val,  _, _ = build_feature_matrix(df_val,  CONFIG, fit_mode=False, artifacts=artifacts)
    X_test, _, _ = build_feature_matrix(df_test, CONFIG, fit_mode=False, artifacts=artifacts)
    LOGGER.info('Feature shapes — train: %s, val: %s, test: %s', X_train.shape, X_val.shape, X_test.shape)
    LOGGER.info('Feature names (%d): %s', len(feature_names), feature_names)
    with open('model_5_prep_artifacts.pkl', 'wb') as f:
        pickle.dump({'prep_artifacts': artifacts, 'feature_names': feature_names}, f)
    LOGGER.info('Preprocessing artifacts saved')

    # 6. ET labels
    et_train = df_train['log_emergence_sec'].values.astype(np.float32)
    et_val   = df_val['log_emergence_sec'].values.astype(np.float32)
    et_test  = df_test['log_emergence_sec'].values.astype(np.float32)
    LOGGER.info('Valid ET labels — train: %d/%d, val: %d/%d, test: %d/%d',
                (~np.isnan(et_train)).sum(), len(et_train),
                (~np.isnan(et_val)).sum(),   len(et_val),
                (~np.isnan(et_test)).sum(),  len(et_test))

    # 7. Datasets + DataLoaders
    train_ds = Model5Dataset(X_train, df_train['caseid'].tolist(), et_train)
    val_ds   = Model5Dataset(X_val,   df_val['caseid'].tolist(),   et_val)
    test_ds  = Model5Dataset(X_test,  df_test['caseid'].tolist(),  et_test)
    train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, CONFIG)

    # 8. Model
    model = PatientContextMLP(X_train.shape[1], CONFIG['hidden_dims'], CONFIG['embedding_dim'], CONFIG['dropout'])
    LOGGER.info('PatientContextMLP — input_dim=%d, params=%d', X_train.shape[1], sum(p.numel() for p in model.parameters() if p.requires_grad))

    # 9-12. Train → Evaluate → SHAP → Plot
    model = train_model(model, train_loader, val_loader, device, CONFIG)
    evaluate_test(model, test_loader, device, CONFIG)
    explain_with_shap(model, test_loader, feature_names, device, CONFIG)
    plot_predictions(model, test_loader, device, CONFIG)

    LOGGER.info('══════════════════════════════════════════════════════════')
    LOGGER.info('Model 5 pipeline complete.')
    LOGGER.info('══════════════════════════════════════════════════════════')
    del train_ds, val_ds, test_ds, X_train, X_val, X_test; gc.collect()


if __name__ == '__main__':
    main()
